# Airline Ticket Price Prediction — Final Colab Notebook

**Beginner-friendly practical notebook** for cleaning, visualization, statistics, and ticket-price prediction.

**Dataset:** Indian Flight Price Prediction dataset: https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction

Upload the extracted CSV when prompted. The dataset contains historical prices, not live fares. The prediction model now uses **source city** and **destination city** along with flight duration, days left, and number of stops.

## Load the airline dataset

In [ ]:
from google.colab import files

uploaded = files.upload()  # Choose the airline CSV file
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Rows and columns:", df.shape)
print(df.head())

##  — Data cleaning and descriptive statistics

In [ ]:
# See missing values and duplicate rows
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates()

# Remove rows where ticket price is missing
df = df.dropna(subset=["price"])

# Basic statistics for ticket price
print("Mean:", df["price"].mean())
print("Median:", df["price"].median())
print("Mode:", df["price"].mode()[0])
print("Variance:", df["price"].var())
print("Standard deviation:", df["price"].std())
print("Quartiles:")
print(df["price"].quantile([0.25, 0.50, 0.75]))

# Summary statistics
print(df.describe())

##  — Data visualization

In [ ]:
import matplotlib.pyplot as plt

# 1. Histogram
plt.hist(df["price"], bins=30)
plt.title("Distribution of Ticket Prices")
plt.xlabel("Ticket price")
plt.ylabel("Number of flights")
plt.show()

# 2. Box plot
plt.boxplot(df["price"])
plt.title("Ticket Price Outliers")
plt.ylabel("Ticket price")
plt.show()

# 3. Bar chart: average price by airline
df.groupby("airline")["price"].mean().plot(kind="bar")
plt.title("Average Price by Airline")
plt.ylabel("Average price")
plt.show()

# 4. Pie chart: number of flights by airline
df["airline"].value_counts().plot(kind="pie", autopct="%1.1f%%")
plt.title("Flights by Airline")
plt.ylabel("")
plt.show()

# 5. Scatter plot: days left vs price
plt.scatter(df["days_left"], df["price"], alpha=0.3)
plt.title("Days Left vs Ticket Price")
plt.xlabel("Days left before departure")
plt.ylabel("Ticket price")
plt.show()

# 6. Line chart: average price by days left
df.groupby("days_left")["price"].mean().sort_index().plot(kind="line")
plt.title("Average Ticket Price by Days Left")
plt.xlabel("Days left")
plt.ylabel("Average price")
plt.show()

## — Probability and sampling

In [ ]:
import numpy as np

# Probability that a randomly selected ticket costs more than the mean
mean_price = df["price"].mean()
probability = (df["price"] > mean_price).mean()
print("Probability price is above the mean:", probability)

# Random sample of 10 flights
random_sample = df.sample(n=10, random_state=1)
print(random_sample[["airline", "price"]])

# Sample ticket prices and view their distribution
plt.hist(random_sample["price"], bins=5)
plt.title("Prices in a Random Sample")
plt.xlabel("Ticket price")
plt.show()

##  — Central Limit Theorem and point estimation

In [ ]:
# Take 100 samples and calculate each sample's mean price
sample_means = []

for i in range(100):
    sample = df["price"].sample(n=30, replace=True, random_state=i)
    sample_means.append(sample.mean())

print("Population mean estimate:", df["price"].mean())
print("Mean of sample means:", np.mean(sample_means))

plt.hist(sample_means, bins=15)
plt.title("Central Limit Theorem: Sample Means")
plt.xlabel("Sample mean price")
plt.ylabel("Frequency")
plt.show()

##  — T-test
We compare ticket prices for nonstop and connecting flights. The code checks the dataset's `stops` values and assumes `zero` means nonstop, as in the common Kaggle dataset.

In [ ]:
from scipy.stats import ttest_ind

print(df["stops"].value_counts())

# Separate nonstop and connecting flight prices
nonstop = df[df["stops"].astype(str).str.lower() == "zero"]["price"]
connecting = df[df["stops"].astype(str).str.lower() != "zero"]["price"]

# Welch's independent t-test
t_value, p_value = ttest_ind(nonstop, connecting, equal_var=False)

print("Average nonstop price:", nonstop.mean())
print("Average connecting price:", connecting.mean())
print("T statistic:", t_value)
print("P value:", p_value)

if p_value < 0.05:
    print("There is a statistically significant difference in average prices.")
else:
    print("There is not enough evidence of a difference in average prices.")

##   — Multiple Linear Regression with Source and Destination City

This model predicts ticket price using five inputs: flight duration, days left before departure, number of stops, source city, and destination city. City names are text, so we use **One-Hot Encoding** to convert them into model-friendly numeric features. A pipeline keeps the preprocessing and model together.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Convert duration such as '2h 50m' into minutes
def convert_duration(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    hours = 0
    minutes = 0
    for part in text.split():
        if "h" in part:
            try:
                hours = int(part.replace("h", ""))
            except ValueError:
                pass
        elif "m" in part:
            try:
                minutes = int(part.replace("m", ""))
            except ValueError:
                pass
    try:
        return float(text) if text.isdigit() else hours * 60 + minutes
    except ValueError:
        return np.nan

# Convert duration and stops into numbers
df["duration_minutes"] = df["duration"].apply(convert_duration)
df["stops_number"] = df["stops"].astype(str).str.lower().map({
    "zero": 0, "one": 1, "two_or_more": 2,
    "non-stop": 0, "nonstop": 0, "0": 0, "1": 1, "2": 2
})

# Check that the dataset contains the city columns needed by the model
required_columns = ["source_city", "destination_city"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(
        "Missing required column(s): " + ", ".join(missing_columns) +
        ". Please upload a flight dataset with source_city and destination_city."
    )

# Prepare a clean dataset for the model
model_data = df[[
    "duration_minutes", "days_left", "stops_number",
    "source_city", "destination_city", "price"
]].copy()
model_data["days_left"] = pd.to_numeric(model_data["days_left"], errors="coerce")
model_data["price"] = pd.to_numeric(model_data["price"], errors="coerce")
model_data["source_city"] = model_data["source_city"].astype("string").str.strip()
model_data["destination_city"] = model_data["destination_city"].astype("string").str.strip()
model_data = model_data.dropna(subset=[
    "duration_minutes", "days_left", "stops_number", "source_city",
    "destination_city", "price"
])

# X = input features; y = the ticket price we want to predict
X = model_data[[
    "duration_minutes", "days_left", "stops_number",
    "source_city", "destination_city"
]]
y = model_data["price"]

numeric_features = ["duration_minutes", "days_left", "stops_number"]
city_features = ["source_city", "destination_city"]

# Fill missing numeric values and convert city names with One-Hot Encoding
preprocessor = ColumnTransformer(transformers=[
    ("numbers", SimpleImputer(strategy="median"), numeric_features),
    ("cities", Pipeline(steps=[
        ("fill_missing_city", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore"))
    ]), city_features)
])

# Pipeline performs preprocessing first, then trains Linear Regression
model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regression", LinearRegression())
])

# Split data into training and testing parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train and test the model
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Rows used for model:", len(model_data))
print("Mean Absolute Error (MAE):", round(mean_absolute_error(y_test, predictions), 2))
print("R2 score:", round(r2_score(y_test, predictions), 3))

# Show a few actual prices beside the predicted prices
results = pd.DataFrame({"Actual price": y_test.values, "Predicted price": predictions})
print(results.head())

##  — Streamlit prediction interface



```bash
python -m pip install -r requirements.txt
python -m streamlit run app.py
```

Streamlit does not reliably display inside a normal Colab cell; run it as a separate local app or deploy it to Streamlit Community Cloud.

In [ ]:
# This cell is only for installing the Streamlit package in Colab, if needed.
# The app itself is run from app.py in a local terminal, not inside this notebook.
!pip -q install streamlit scikit-learn pandas numpy